#  Spatiotemporal Analysis of Deforestation Using Siamese U-Net v5
**Binary Change Detection for Forest Loss | Meghalaya & Nagaland, Northeast India (2021 → 2023)**

## 0. Setup & Reproducibility

In [ ]:
import os, random, time, io
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import ipywidgets as widgets

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, f1_score
)
from tensorflow.keras import layers, Model
from IPython.display import display, clear_output

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
tf.keras.mixed_precision.set_global_policy("float32")

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {len(gpus)}")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

## 1. Dataset Path Discovery

In [ ]:
base_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "t1" in dirs and "t2" in dirs and "label" in dirs:
        base_path = os.path.dirname(root)
        print(f"Found dataset at: {base_path}")
        break

if base_path is None:
    base_path = "/kaggle/input/datasets/suranjandas1990/forest-loss-dataset/dataset"
    print(f"Using fallback: {base_path}")

## 2. Config

In [ ]:
CFG = {
    "base_path"       : base_path,
    "regions"         : ["Meghalaya_2021_2023", "Nagaland_2021_2023"],
    "patch_size"      : 256,
    "in_channels"     : 6,
    "use_ndvi"        : True,
    "use_evi_savi"    : True,
    "use_diff_channel": True,
    # Training
    "batch_size"      : 8,
    "epochs"          : 80,
    "lr"              : 1e-3,
    "warmup_epochs"   : 10,
    "dropout"         : 0.3,
    "grad_clip"       : 1.0,
    # Loss weights
    "pos_weight"      : 5.0,
    "label_smoothing" : 0.00,
    "bce_weight"      : 0.40,
    "lovasz_weight"   : 0.35,
    "boundary_weight" : 0.10,
    "ohem_weight"     : 0.05,
    "ohem_k"          : 256,
    # Evaluation
    "threshold"       : 0.5,
    "use_tta"         : True,
    "pixel_res_m"     : 30,
    "seed"            : SEED,
    "output_dir"      : "/kaggle/working/",
}

SPECTRAL_CHANNELS = CFG["in_channels"] + \
                    (1 if CFG["use_ndvi"] else 0) + \
                    (2 if CFG["use_evi_savi"] else 0)
DIFF_CHANNELS  = SPECTRAL_CHANNELS if CFG["use_diff_channel"] else 0
MODEL_CHANNELS = SPECTRAL_CHANNELS

os.makedirs(CFG["output_dir"], exist_ok=True)
print(f"Spectral channels per timestamp : {SPECTRAL_CHANNELS}")
print(f"Outputs will save to            : {CFG['output_dir']}")

## 3. Path Verification

In [ ]:
print("Verifying dataset structure...")
for region in CFG["regions"]:
    for split in ["t1", "t2", "label"]:
        p = os.path.join(CFG["base_path"], region, split)
        if os.path.exists(p):
            n = len([f for f in os.listdir(p) if f.endswith(".npy")])
            print(f"  OK  {region}/{split}: {n} files")
        else:
            print(f"  MISSING: {p}")

## 4. File Discovery & Stratified Split

In [ ]:
def discover_files(base_path, regions):
    t1_files, t2_files, label_files, region_labels = [], [], [], []
    for region in regions:
        t1_dir    = os.path.join(base_path, region, "t1")
        t2_dir    = os.path.join(base_path, region, "t2")
        label_dir = os.path.join(base_path, region, "label")
        for fname in sorted(os.listdir(t1_dir)):
            if fname.endswith(".npy"):
                t1_files.append(os.path.join(t1_dir, fname))
                t2_files.append(os.path.join(t2_dir, fname))
                label_files.append(os.path.join(label_dir, fname))
                region_labels.append(region)
    return t1_files, t2_files, label_files, region_labels

t1_all, t2_all, y_all, reg_all = discover_files(CFG["base_path"], CFG["regions"])
print(f"Total patches: {len(t1_all)}")
for r in CFG["regions"]:
    print(f"  {r}: {reg_all.count(r)} patches")

print("\nComputing stratification labels (may take ~30s)...")
indices = list(range(len(t1_all)))
strat   = [1 if np.load(y_all[i]).mean() > 0.01 else 0 for i in indices]

train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=SEED, stratify=strat
)

train_t1 = [t1_all[i] for i in train_idx]
train_t2 = [t2_all[i] for i in train_idx]
train_y  = [y_all[i]  for i in train_idx]
val_t1   = [t1_all[i] for i in val_idx]
val_t2   = [t2_all[i] for i in val_idx]
val_y    = [y_all[i]  for i in val_idx]
val_reg  = [reg_all[i] for i in val_idx]

print(f"\nTrain: {len(train_t1)} | Val: {len(val_t1)}")
for r in CFG["regions"]:
    tr = sum(1 for i in train_idx if reg_all[i] == r)
    va = sum(1 for i in val_idx   if reg_all[i] == r)
    print(f"  {r.split('_')[0]:12s} → train: {tr} | val: {va}")

## 5. Compute Global Normalisation Stats

In [ ]:
import random as pyrandom

print("Computing global normalisation stats from 500 random patches...")
sample_means, sample_stds = [], []

sample_paths = pyrandom.sample(t1_all + t2_all, min(500, len(t1_all + t2_all)))

for path in sample_paths:
    img = np.load(path).astype(np.float32)
    if img.ndim == 3 and img.shape[0] == CFG["in_channels"]:
        img = np.transpose(img, (1, 2, 0))
    sample_means.append(img.mean(axis=(0, 1)))
    sample_stds.append(img.std(axis=(0, 1)))

GLOBAL_STATS = {
    "mean": np.mean(sample_means, axis=0).astype(np.float32),
    "std":  np.mean(sample_stds,  axis=0).astype(np.float32),
}

print(f"mean : {GLOBAL_STATS['mean']}")
print(f"std  : {GLOBAL_STATS['std']}")
print("Global stats ready.")

## 5b. Class Imbalance Analysis

In [ ]:
def compute_class_balance(label_files, sample_n=200):
    fractions = []
    indices   = np.random.choice(len(label_files), min(sample_n, len(label_files)),
                                 replace=False)
    for i in indices:
        mask = np.load(label_files[i])
        fractions.append(mask.mean())
    mean_frac = np.mean(fractions)
    print(f"Mean forest-loss pixel fraction : {mean_frac:.4f} ({mean_frac*100:.2f}%)")
    print(f"Suggested pos_weight            : {(1-mean_frac)/(mean_frac+1e-6):.1f}  "
          f"(set to {CFG['pos_weight']})")
    return mean_frac

_ = compute_class_balance(y_all)

## 6. Data Loading & Preprocessing

In [ ]:
def add_vegetation_indices(img):
    """
    Append NDVI, EVI, SAVI to a (H, W, 6) image.
    Applied AFTER global normalisation.
    Band order: [B, G, R, NIR, SWIR1, SWIR2]
    """
    nir  = img[..., 3:4]
    red  = img[..., 2:3]
    blue = img[..., 0:1]

    ndvi = (nir - red) / (nir + red + 1e-6)
    ndvi = np.clip(ndvi, -1, 1)

    evi  = 2.5 * (nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0 + 1e-6)
    evi  = np.clip(evi, -1, 1)

    L    = 0.5
    savi = ((nir - red) / (nir + red + L + 1e-6)) * (1.0 + L)
    savi = np.clip(savi, -1, 1)

    return np.concatenate([img, ndvi, evi, savi], axis=-1)


def normalize_image(img):
    img  = np.clip(img, 0, 1)
    mean = GLOBAL_STATS["mean"]
    std  = GLOBAL_STATS["std"]
    return (img - mean) / (std + 1e-6)


def load_and_preprocess(t1_path, t2_path, y_path):

    def _parse(t1_path, t2_path, y_path):
        img1 = np.load(t1_path.numpy().decode()).astype(np.float32)
        img2 = np.load(t2_path.numpy().decode()).astype(np.float32)
        mask = np.load(y_path.numpy().decode()).astype(np.float32)

        if img1.ndim == 3 and img1.shape[0] == CFG["in_channels"]:
            img1 = np.transpose(img1, (1, 2, 0))
            img2 = np.transpose(img2, (1, 2, 0))

        img1 = normalize_image(img1)
        img2 = normalize_image(img2)

        img1 = add_vegetation_indices(img1)
        img2 = add_vegetation_indices(img2)

        diff = np.abs(img2 - img1)

        mask = (mask > 0).astype(np.float32)
        if mask.ndim == 2:
            mask = np.expand_dims(mask, axis=-1)

        return img1, img2, diff, mask

    img1, img2, diff, mask = tf.py_function(
        _parse,
        [t1_path, t2_path, y_path],
        [tf.float32, tf.float32, tf.float32, tf.float32]
    )

    n_ch = CFG["in_channels"] + \
           (1 if CFG["use_ndvi"] else 0) + \
           (2 if CFG["use_evi_savi"] else 0)

    img1.set_shape((CFG["patch_size"], CFG["patch_size"], n_ch))
    img2.set_shape((CFG["patch_size"], CFG["patch_size"], n_ch))
    diff.set_shape((CFG["patch_size"], CFG["patch_size"], n_ch))
    mask.set_shape((CFG["patch_size"], CFG["patch_size"], 1))

    return (img1, img2, diff), mask

## 7. Augmentation

In [ ]:
def augment(inputs, mask):
    img_t1, img_t2, diff = inputs

    if tf.random.uniform(()) > 0.5:
        img_t1 = tf.image.flip_left_right(img_t1)
        img_t2 = tf.image.flip_left_right(img_t2)
        diff   = tf.image.flip_left_right(diff)
        mask   = tf.image.flip_left_right(mask)

    if tf.random.uniform(()) > 0.5:
        img_t1 = tf.image.flip_up_down(img_t1)
        img_t2 = tf.image.flip_up_down(img_t2)
        diff   = tf.image.flip_up_down(diff)
        mask   = tf.image.flip_up_down(mask)

    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img_t1 = tf.image.rot90(img_t1, k)
    img_t2 = tf.image.rot90(img_t2, k)
    diff   = tf.image.rot90(diff, k)
    mask   = tf.image.rot90(mask, k)

    if tf.random.uniform(()) > 0.5:
        img_t1 += tf.random.normal(tf.shape(img_t1), stddev=0.01)
        img_t2 += tf.random.normal(tf.shape(img_t2), stddev=0.01)

    scale  = tf.random.uniform([], 0.95, 1.05)
    img_t1 = img_t1 * scale
    img_t2 = img_t2 * scale

    diff = tf.abs(img_t2 - img_t1)

    return (img_t1, img_t2, diff), mask

## 8. tf.data Pipelines

In [ ]:
def create_balanced_ds(t1_list, t2_list, y_list):
    pos_indices, neg_indices = [], []
    print("Analysing label density for balanced sampling...")
    for i, path in enumerate(y_list):
        m = np.load(path)
        if np.sum(m) > 10:
            pos_indices.append(i)
        else:
            neg_indices.append(i)
    print(f"  Positive patches: {len(pos_indices)} | Negative: {len(neg_indices)}")

    pos_ds = tf.data.Dataset.from_tensor_slices((
        [t1_list[i] for i in pos_indices],
        [t2_list[i] for i in pos_indices],
        [y_list[i]  for i in pos_indices],
    )).repeat()

    neg_ds = tf.data.Dataset.from_tensor_slices((
        [t1_list[i] for i in neg_indices],
        [t2_list[i] for i in neg_indices],
        [y_list[i]  for i in neg_indices],
    ))

    return tf.data.Dataset.sample_from_datasets(
        [pos_ds, neg_ds], weights=[0.5, 0.5]
    )


train_ds = (
    create_balanced_ds(train_t1, train_t2, train_y)
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(200)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(CFG["batch_size"])
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_t1, val_t2, val_y))
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(CFG["batch_size"])
    .prefetch(tf.data.AUTOTUNE)
)

print("tf.data pipelines ready.")

## 9. Model: Siamese U-Net v4

In [ ]:
from tensorflow.keras.layers import GroupNormalization

def conv_block(filters, dropout_rate=0.0, name=None):
    block = tf.keras.Sequential(name=name)
    block.add(layers.Conv2D(filters, 3, padding="same", use_bias=False))
    block.add(GroupNormalization(groups=min(8, filters)))
    block.add(layers.Activation("gelu"))
    block.add(layers.Conv2D(filters, 3, padding="same", use_bias=False))
    block.add(GroupNormalization(groups=min(8, filters)))
    block.add(layers.Activation("gelu"))
    if dropout_rate > 0:
        block.add(layers.Dropout(dropout_rate))
    return block


def channel_attention(x, ratio=8):
    ch = x.shape[-1]
    sq = layers.GlobalAveragePooling2D()(x)
    sq = layers.Dense(max(ch // ratio, 1), activation="gelu")(sq)
    sq = layers.Dense(ch, activation="sigmoid")(sq)
    sq = layers.Reshape((1, 1, ch))(sq)
    return layers.Multiply()([x, sq])


def aspp_block(x, filters=256):
    b0 = layers.Conv2D(filters, 1, padding="same", use_bias=False)(x)
    b0 = GroupNormalization(groups=min(8, filters))(b0)
    b0 = layers.Activation("gelu")(b0)

    b1 = layers.Conv2D(filters, 3, dilation_rate=4,  padding="same", use_bias=False)(x)
    b1 = GroupNormalization(groups=min(8, filters))(b1)
    b1 = layers.Activation("gelu")(b1)

    b2 = layers.Conv2D(filters, 3, dilation_rate=8,  padding="same", use_bias=False)(x)
    b2 = GroupNormalization(groups=min(8, filters))(b2)
    b2 = layers.Activation("gelu")(b2)

    b3 = layers.Conv2D(filters, 3, dilation_rate=12, padding="same", use_bias=False)(x)
    b3 = GroupNormalization(groups=min(8, filters))(b3)
    b3 = layers.Activation("gelu")(b3)

    gp = layers.GlobalAveragePooling2D()(x)
    gp = layers.Reshape((1, 1, x.shape[-1]))(gp)
    gp = layers.Conv2D(filters, 1, use_bias=False)(gp)
    gp = GroupNormalization(groups=min(8, filters))(gp)
    gp = layers.Activation("gelu")(gp)
    gp = layers.Lambda(
        lambda t: tf.image.resize(t[0], (tf.shape(t[1])[1], tf.shape(t[1])[2]))
    )([gp, x])

    out = layers.Concatenate()([b0, b1, b2, b3, gp])
    out = layers.Conv2D(filters, 1, padding="same", use_bias=False)(out)
    out = GroupNormalization(groups=min(8, filters))(out)
    return layers.Activation("gelu")(out)


def transformer_bottleneck(x, filters=256, num_heads=4):
    H = CFG["patch_size"] // 16
    W = CFG["patch_size"] // 16

    seq  = layers.Reshape((H * W, x.shape[-1]))(x)
    qkv  = layers.Dense(filters * 3, use_bias=False)(seq)
    q    = qkv[..., :filters]
    k    = qkv[..., filters:filters*2]
    v    = qkv[..., filters*2:]

    head_dim = filters // num_heads

    def split_heads(t):
        t = layers.Reshape((H * W, num_heads, head_dim))(t)
        return tf.transpose(t, [0, 2, 1, 3])

    q    = layers.Lambda(split_heads)(q)
    k    = layers.Lambda(split_heads)(k)
    v    = layers.Lambda(split_heads)(v)

    scale = tf.math.sqrt(tf.cast(head_dim, tf.float32))
    attn  = layers.Lambda(
        lambda t: tf.matmul(t[0], t[1], transpose_b=True) / scale
    )([q, k])
    attn  = layers.Softmax(axis=-1)(attn)
    attn  = layers.Dropout(0.1)(attn)

    out   = layers.Lambda(lambda t: tf.matmul(t[0], t[1]))([attn, v])
    out   = layers.Lambda(lambda t: tf.transpose(t, [0, 2, 1, 3]))(out)
    out   = layers.Reshape((H * W, filters))(out)
    out   = layers.Dense(filters)(out)
    out   = layers.Reshape((H, W, filters))(out)

    out   = layers.Add()([out, x])
    out   = layers.LayerNormalization()(out)

    ffn   = layers.Reshape((H * W, filters))(out)
    ffn   = layers.Dense(filters * 4, activation="gelu")(ffn)
    ffn   = layers.Dense(filters)(ffn)
    ffn   = layers.Dropout(CFG["dropout"])(ffn)
    ffn   = layers.Reshape((H, W, filters))(ffn)

    out   = layers.Add()([out, ffn])
    return layers.LayerNormalization()(out)


def build_siamese_unet_v4(input_shape, diff_shape, dropout=0.3):
    inp_t1   = layers.Input(input_shape, name="T1")
    inp_t2   = layers.Input(input_shape, name="T2")
    inp_diff = layers.Input(diff_shape,  name="DIFF")

    pool = lambda x: layers.MaxPooling2D(2)(x)

    enc1 = conv_block(32,  name="enc1")
    enc2 = conv_block(64,  name="enc2")
    enc3 = conv_block(128, name="enc3")
    enc4 = conv_block(256, name="enc4")

    c1_t1 = enc1(inp_t1);  p1_t1 = pool(c1_t1)
    c2_t1 = enc2(p1_t1);   p2_t1 = pool(c2_t1)
    c3_t1 = enc3(p2_t1);   p3_t1 = pool(c3_t1)
    c4_t1 = enc4(p3_t1);   p4_t1 = pool(c4_t1)

    c1_t2 = enc1(inp_t2);  p1_t2 = pool(c1_t2)
    c2_t2 = enc2(p1_t2);   p2_t2 = pool(c2_t2)
    c3_t2 = enc3(p2_t2);   p3_t2 = pool(c3_t2)
    c4_t2 = enc4(p3_t2);   p4_t2 = pool(c4_t2)

    diff_feat = layers.Conv2D(32, 3, strides=2, padding="same")(inp_diff)
    diff_feat = layers.Conv2D(32, 3, strides=2, padding="same")(diff_feat)
    diff_feat = layers.Conv2D(32, 3, strides=2, padding="same")(diff_feat)
    diff_feat = layers.Conv2D(32, 3, strides=2, padding="same")(diff_feat)

    skip1 = channel_attention(layers.Concatenate()([c1_t1, c1_t2]))
    skip2 = channel_attention(layers.Concatenate()([c2_t1, c2_t2]))
    skip3 = channel_attention(layers.Concatenate()([c3_t1, c3_t2]))
    skip4 = channel_attention(layers.Concatenate()([c4_t1, c4_t2]))

    b = layers.Concatenate()([p4_t1, p4_t2, diff_feat])
    b = layers.Conv2D(256, 1, use_bias=False)(b)
    b = GroupNormalization(groups=8)(b)
    b = layers.Activation("gelu")(b)
    b = aspp_block(b, 256)
    b = transformer_bottleneck(b, 256, num_heads=4)

    u4 = layers.Conv2DTranspose(256, 2, strides=2, padding="same")(b)
    u4 = layers.Concatenate()([u4, skip4])
    d4 = conv_block(256)(u4)

    u3 = layers.Conv2DTranspose(128, 2, strides=2, padding="same")(d4)
    u3 = layers.Concatenate()([u3, skip3])
    d3 = conv_block(128)(u3)

    u2 = layers.Conv2DTranspose(64, 2, strides=2, padding="same")(d3)
    u2 = layers.Concatenate()([u2, skip2])
    d2 = conv_block(64)(u2)

    u1 = layers.Conv2DTranspose(32, 2, strides=2, padding="same")(d2)
    u1 = layers.Concatenate()([u1, skip1])
    d1 = conv_block(32)(u1)

    out  = layers.Conv2D(1, 1, activation="sigmoid", name="main_output")(d1)

    aux4 = layers.Conv2D(1, 1)(d4)
    aux4 = layers.UpSampling2D(8)(aux4)
    aux4 = layers.Activation("sigmoid", name="aux4_output")(aux4)

    aux3 = layers.Conv2D(1, 1)(d3)
    aux3 = layers.UpSampling2D(4)(aux3)
    aux3 = layers.Activation("sigmoid", name="aux3_output")(aux3)

    aux2 = layers.Conv2D(1, 1)(d2)
    aux2 = layers.UpSampling2D(2)(aux2)
    aux2 = layers.Activation("sigmoid", name="aux2_output")(aux2)

    return Model(
        [inp_t1, inp_t2, inp_diff],
        [out, aux4, aux3, aux2],
        name="SiameseUNet_v4_FINAL"
    )

## 10. Loss Functions & Metrics

In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    union        = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return (2.0 * intersection + smooth) / (union + smooth)


def iou_score(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    union        = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)


def combined_loss(y_true, y_pred):
    y_true_f  = tf.cast(y_true, tf.float32)
    y_pred_f  = tf.cast(y_pred, tf.float32)

    y_true_sq = tf.squeeze(y_true_f, axis=-1)
    y_pred_sq = tf.squeeze(y_pred_f, axis=-1)
    y_pred_sq = tf.clip_by_value(y_pred_sq, 1e-7, 1.0 - 1e-7)

    bce       = -(y_true_sq * tf.math.log(y_pred_sq) +
                  (1.0 - y_true_sq) * tf.math.log(1.0 - y_pred_sq))
    weight    = 1.0 + CFG["pos_weight"] * y_true_sq
    w_bce     = tf.reduce_mean(bce * weight)

    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union        = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f)
    dice_loss    = 1.0 - (2.0 * intersection + 1e-6) / (union + 1e-6)

    return 0.5 * w_bce + 0.5 * dice_loss

## 11. LR Schedule

In [ ]:
def make_lr_schedule(initial_lr, warmup_epochs, total_epochs):
    def schedule(epoch):
        if epoch < warmup_epochs:
            return initial_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return initial_lr * 0.5 * (1.0 + np.cos(np.pi * progress))
    return schedule

## 12. Compile

In [ ]:
import gc
gc.collect()
tf.keras.backend.clear_session()

SPECTRAL_CHANNELS = CFG["in_channels"] + \
                    (1 if CFG["use_ndvi"] else 0) + \
                    (2 if CFG["use_evi_savi"] else 0)
print(f"SPECTRAL_CHANNELS = {SPECTRAL_CHANNELS}")

input_shape = (CFG["patch_size"], CFG["patch_size"], SPECTRAL_CHANNELS)
diff_shape  = (CFG["patch_size"], CFG["patch_size"], SPECTRAL_CHANNELS)

model = build_siamese_unet_v4(
    input_shape=input_shape,
    diff_shape=diff_shape,
    dropout=CFG["dropout"]
)
model.summary()
print(f"\nTotal parameters: {model.count_params():,}")


def triple_target(inputs, mask):
    return inputs, {
        "main_output": mask,
        "aux4_output": mask,
        "aux3_output": mask,
        "aux2_output": mask,
    }

train_ds_v4 = train_ds.map(triple_target, num_parallel_calls=tf.data.AUTOTUNE)
val_ds_v4   = val_ds.map(triple_target,   num_parallel_calls=tf.data.AUTOTUNE)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=CFG["lr"],
        clipnorm=CFG["grad_clip"]
    ),
    loss={
        "main_output": combined_loss,
        "aux4_output": combined_loss,
        "aux3_output": combined_loss,
        "aux2_output": combined_loss,
    },
    loss_weights={
        "main_output": 1.0,
        "aux4_output": 0.2,
        "aux3_output": 0.4,
        "aux2_output": 0.2,
    },
    metrics={
        "main_output": [dice_coef, iou_score]
    }
)
print("Model compiled successfully.")

## 13. Train

In [ ]:
callbacks_v4 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_main_output_dice_coef",
        patience=20,
        mode="max",
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.LearningRateScheduler(
        make_lr_schedule(CFG["lr"], CFG["warmup_epochs"], CFG["epochs"]),
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(CFG["output_dir"], "best_forest_model_v4.keras"),
        monitor="val_main_output_dice_coef",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        os.path.join(CFG["output_dir"], "training_log_v4.csv")
    ),
    # ReduceLROnPlateau REMOVED — conflicts with LearningRateScheduler
]

steps_per_epoch  = len(train_t1) // CFG["batch_size"]
validation_steps = None  # evaluate full val set every epoch

print(f"steps_per_epoch : {steps_per_epoch}")
print(f"validation_steps: full dataset (None)")

t0 = time.time()
history_v4 = model.fit(
    train_ds_v4,
    validation_data=val_ds_v4,
    epochs=CFG["epochs"],
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks_v4,
)
elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed/60:.1f} minutes")

model.save(os.path.join(CFG["output_dir"], "forest_loss_final_model_v4.keras"))
pd.DataFrame(history_v4.history).to_csv(
    os.path.join(CFG["output_dir"], "training_history_v4.csv"), index=False
)
print("Model and history saved.")

## 14. Training Curves

In [ ]:
def plot_training_curves_v4(history, save_dir):
    h      = history.history
    epochs = range(1, len(h["loss"]) + 1)

    pairs = [
        ("loss",                  "val_loss",                  "Combined Loss",    "Loss"),
        ("main_output_dice_coef", "val_main_output_dice_coef", "Dice Coefficient", "Dice"),
        ("main_output_iou_score", "val_main_output_iou_score", "IoU Score",        "IoU"),
        ("aux3_output_loss",      "val_aux3_output_loss",      "Aux3 Loss",        "Loss"),
    ]
    pairs = [(tr, va, title, yl) for tr, va, title, yl in pairs if tr in h and va in h]

    n    = len(pairs)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1: axes = [axes]

    for ax, (tr, va, title, ylabel) in zip(axes, pairs):
        ax.plot(epochs, h[tr], label="Train",      color="#185FA5", linewidth=2)
        ax.plot(epochs, h[va], label="Validation", color="#D85A30", linewidth=2,
                linestyle="--")
        ax.set_title(title, fontsize=13)
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(alpha=0.3)

    plt.suptitle(
        "Forest Change Detection v4 — Training Performance\n"
        "Siamese U-Net + ASPP + Transformer | Meghalaya & Nagaland 2021→2023",
        fontsize=14, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_curves_v4.png"),
                dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved: training_curves_v4.png")

plot_training_curves_v4(history_v4, CFG["output_dir"])

## 15. Test-Time Augmentation

In [ ]:
def tta_predict(model, t1_batch, t2_batch, diff_batch, use_tta=True):
    def get_main(preds):
        return preds[0] if isinstance(preds, (list, tuple)) else preds

    t1_np   = t1_batch.numpy()   if hasattr(t1_batch,   "numpy") else np.array(t1_batch)
    t2_np   = t2_batch.numpy()   if hasattr(t2_batch,   "numpy") else np.array(t2_batch)
    diff_np = diff_batch.numpy() if hasattr(diff_batch, "numpy") else np.array(diff_batch)

    if not use_tta:
        return get_main(model.predict([t1_np, t2_np, diff_np], verbose=0))

    preds_sum  = np.zeros(
        (t1_np.shape[0], CFG["patch_size"], CFG["patch_size"], 1), dtype=np.float32
    )
    transforms = [
        (False, False, 0), (True,  False, 0), (False, True,  0), (True,  True,  0),
        (False, False, 1), (True,  False, 1), (False, True,  1), (True,  True,  1),
    ]
    for flip_h, flip_v, rot in transforms:
        t1 = t1_np.copy(); t2 = t2_np.copy(); df = diff_np.copy()
        if flip_h: t1=t1[:,:,::-1,:]; t2=t2[:,:,::-1,:]; df=df[:,:,::-1,:]
        if flip_v: t1=t1[:,::-1,:,:]; t2=t2[:,::-1,:,:]; df=df[:,::-1,:,:]
        if rot:
            t1 = np.rot90(t1, k=1, axes=(1,2))
            t2 = np.rot90(t2, k=1, axes=(1,2))
            df = np.rot90(df, k=1, axes=(1,2))
        p = get_main(model.predict([t1, t2, df], verbose=0))
        if rot:    p = np.rot90(p, k=-1, axes=(1,2))
        if flip_v: p = p[:,::-1,:,:]
        if flip_h: p = p[:,:,::-1,:]
        preds_sum += p

    return preds_sum / len(transforms)

## 16. Full Evaluation

In [ ]:
def evaluate_model(model, dataset, threshold=CFG["threshold"]):
    print(f"Running evaluation {'with TTA' if CFG['use_tta'] else 'without TTA'}...")
    y_true_all, y_soft_all = [], []

    for inputs, targets in dataset:
        t1b, t2b, diffb = inputs
        mb    = targets["main_output"] if isinstance(targets, dict) else targets
        preds = tta_predict(model, t1b, t2b, diffb, use_tta=CFG["use_tta"])
        y_soft_all.extend(preds.flatten())
        y_true_all.extend(mb.numpy().flatten())

    y_true = np.array(y_true_all, dtype=np.uint8)
    y_soft = np.array(y_soft_all, dtype=np.float32)
    y_pred = (y_soft > threshold).astype(np.uint8)

    print("\n── Classification Report ──────────────────────────────")
    print(classification_report(
        y_true, y_pred,
        target_names=["No forest loss", "Forest loss"],
        digits=4
    ))
    auc = roc_auc_score(y_true, y_soft)
    print(f"ROC-AUC: {auc:.4f}")
    return y_true, y_soft, y_pred

y_true_val, y_soft_val, y_pred_val = evaluate_model(model, val_ds_v4)

## 17. Visualisation Suite

In [ ]:
def plot_confusion_matrix(y_true, y_pred, save_dir):
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
                xticklabels=["No loss", "Forest loss"],
                yticklabels=["No loss", "Forest loss"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix — Full Validation Set (v4)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "confusion_matrix_v4.png"), dpi=150)
    plt.show()

plot_confusion_matrix(y_true_val, y_pred_val, CFG["output_dir"])


def plot_roc_curve(y_true, y_soft, save_dir):
    fpr, tpr, _ = roc_curve(y_true, y_soft)
    auc = roc_auc_score(y_true, y_soft)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color="#185FA5", linewidth=2, label=f"AUC = {auc:.4f}")
    plt.plot([0,1],[0,1], "k--", linewidth=1)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve — Forest Loss Detection v4")
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "roc_curve_v4.png"), dpi=150)
    plt.show()

plot_roc_curve(y_true_val, y_soft_val, CFG["output_dir"])


def plot_threshold_sweep(y_true, y_soft, save_dir):
    thresholds = np.linspace(0.1, 0.9, 50)
    f1s, ious  = [], []
    for t in thresholds:
        y_bin = (y_soft > t).astype(np.uint8)
        f1s.append(f1_score(y_true, y_bin, zero_division=0))
        inter = np.logical_and(y_true, y_bin).sum()
        union = np.logical_or(y_true,  y_bin).sum()
        ious.append(inter / (union + 1e-6))
    best_t = thresholds[np.argmax(f1s)]
    print(f"Best threshold by F1: {best_t:.2f}  (F1={max(f1s):.4f})")
    plt.figure(figsize=(8, 4))
    plt.plot(thresholds, f1s,  label="F1",  color="#185FA5", linewidth=2)
    plt.plot(thresholds, ious, label="IoU", color="#1D9E75", linewidth=2)
    plt.axvline(best_t, color="#D85A30", linestyle="--", label=f"Best={best_t:.2f}")
    plt.xlabel("Threshold"); plt.title("F1 & IoU vs Decision Threshold (v4)")
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "threshold_sweep_v4.png"), dpi=150)
    plt.show()
    return best_t

best_threshold = plot_threshold_sweep(y_true_val, y_soft_val, CFG["output_dir"])


def per_region_evaluation(model, val_t1, val_t2, val_y, val_reg, threshold):
    print("\n── Per-Region Breakdown ───────────────────────────────")
    for region in sorted(set(val_reg)):
        indices = [i for i, r in enumerate(val_reg) if r == region]
        r_ds = (
            tf.data.Dataset.from_tensor_slices((
                [val_t1[i] for i in indices],
                [val_t2[i] for i in indices],
                [val_y[i]  for i in indices],
            ))
            .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(CFG["batch_size"]).prefetch(tf.data.AUTOTUNE)
            .map(triple_target, num_parallel_calls=tf.data.AUTOTUNE)
        )
        yt, ys = [], []
        for inputs, targets in r_ds:
            t1b, t2b, diffb = inputs
            preds = tta_predict(model, t1b, t2b, diffb, use_tta=CFG["use_tta"])
            ys.extend(preds.flatten())
            yt.extend(targets["main_output"].numpy().flatten())
        yt    = np.array(yt, dtype=np.uint8)
        ys    = np.array(ys)
        yp    = (ys > threshold).astype(np.uint8)
        inter = np.logical_and(yt, yp).sum()
        union = np.logical_or(yt,  yp).sum()
        dice  = 2 * inter / (yt.sum() + yp.sum() + 1e-6)
        iou   = inter / (union + 1e-6)
        f1    = f1_score(yt, yp, zero_division=0)
        print(f"  {region.split('_')[0]:12s} → "
              f"Dice: {dice:.4f} | IoU: {iou:.4f} | F1: {f1:.4f} | n={len(indices)}")

per_region_evaluation(model, val_t1, val_t2, val_y, val_reg, best_threshold)

## 18. Qualitative Visualisation

In [ ]:
def make_rgb_composite(img, bands=(2, 1, 0)):
    rgb = img[..., list(bands)].copy().astype(np.float32)
    for i in range(3):
        p2, p98 = np.percentile(rgb[..., i], (2, 98))
        rgb[..., i] = np.clip((rgb[..., i] - p2) / (p98 - p2 + 1e-6), 0, 1)
    return rgb


def visualize_predictions(model, dataset, save_dir, n=4, threshold=0.5):
    fig, axes = plt.subplots(n, 5, figsize=(28, 5.5 * n))
    for i, (inputs, targets) in enumerate(dataset.take(n)):
        t1b, t2b, diffb = inputs
        mb        = targets["main_output"] if isinstance(targets, dict) else targets
        t1_np     = t1b[0].numpy()
        t2_np     = t2b[0].numpy()
        gt        = mb[0].numpy()[:,:,0].astype(np.uint8)
        pred_raw  = tta_predict(model, t1b[:1], t2b[:1], diffb[:1],
                                use_tta=CFG["use_tta"])[0,:,:,0]
        pred_mask = (pred_raw > threshold).astype(np.uint8)
        rgb_t1    = make_rgb_composite(t1_np, bands=(2,1,0))
        fcc_t2    = make_rgb_composite(t2_np, bands=(3,2,1))
        inter     = np.logical_and(gt, pred_mask).sum()
        dice      = 2 * inter / (gt.sum() + pred_mask.sum() + 1e-6)
        area_ha   = pred_mask.sum() * (CFG["pixel_res_m"] ** 2) / 10_000

        axes[i,0].imshow(rgb_t1);   axes[i,0].set_title("T1 2021 (True colour)")
        axes[i,1].imshow(fcc_t2);   axes[i,1].set_title("T2 2023 (False-colour IR)")
        axes[i,2].imshow(gt,        cmap="Greens", vmin=0, vmax=1)
        axes[i,2].set_title("Ground truth")
        axes[i,3].imshow(pred_raw,  cmap="hot",    vmin=0, vmax=1)
        axes[i,3].set_title("Soft probability")
        axes[i,4].imshow(pred_mask, cmap="Reds",   vmin=0, vmax=1)
        c = "darkgreen" if dice > 0.3 else "darkred"
        axes[i,4].set_title(
            f"Prediction | Dice={dice:.3f}\n{area_ha:.2f} ha detected", color=c)
        for ax in axes[i]: ax.axis("off")

    plt.suptitle(
        "Siamese U-Net v4 — Forest Loss Detection Results\n"
        "SDG 15.2: Halt deforestation monitoring",
        fontsize=15, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "predictions_final_v4.png"),
                dpi=200, bbox_inches="tight")
    plt.show()

visualize_predictions(model, val_ds_v4, CFG["output_dir"],
                      n=4, threshold=best_threshold)


def error_analysis(model, dataset, save_dir, threshold, n=3):
    fp_scores, samples = [], []
    for inputs, targets in dataset:
        t1b, t2b, diffb = inputs
        mb    = targets["main_output"] if isinstance(targets, dict) else targets
        preds = tta_predict(model, t1b, t2b, diffb, use_tta=CFG["use_tta"])
        for j in range(len(preds)):
            gt = mb[j].numpy()[:,:,0].astype(np.uint8)
            pr = (preds[j,:,:,0] > threshold).astype(np.uint8)
            fp = np.logical_and(pr==1, gt==0).sum() / (gt.size + 1e-6)
            fp_scores.append(fp)
            samples.append((t1b[j].numpy(), t2b[j].numpy(), gt, preds[j,:,:,0]))

    fig, axes = plt.subplots(n, 4, figsize=(22, 5*n))
    for row, idx in enumerate(np.argsort(fp_scores)[-n:][::-1]):
        t1_np, t2_np, gt, soft = samples[idx]
        axes[row,0].imshow(make_rgb_composite(t2_np))
        axes[row,0].set_title("T2 RGB")
        axes[row,1].imshow(gt,          cmap="Greens")
        axes[row,1].set_title("Ground truth")
        axes[row,2].imshow(soft>threshold, cmap="Reds")
        axes[row,2].set_title(f"Prediction (FP={fp_scores[idx]:.3f})")
        axes[row,3].imshow(soft,         cmap="hot")
        axes[row,3].set_title("Soft probability")
        for ax in axes[row]: ax.axis("off")
    plt.suptitle("Error Analysis — Highest False Positive Patches",
                 fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "error_analysis_v4.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

error_analysis(model, val_ds_v4, CFG["output_dir"], best_threshold)

## 19. Summary Report

In [ ]:
def print_summary(y_true, y_soft, threshold):
    y_pred = (y_soft > threshold).astype(np.uint8)
    inter  = np.logical_and(y_true, y_pred).sum()
    union  = np.logical_or(y_true,  y_pred).sum()
    dice   = 2 * inter / (y_true.sum() + y_pred.sum() + 1e-6)
    iou    = inter / (union + 1e-6)
    auc    = roc_auc_score(y_true, y_soft)
    f1     = f1_score(y_true, y_pred, zero_division=0)
    print("\n" + "=" * 57)
    print("  FINAL RESULTS — Forest Change Detection Model v4")
    print("=" * 57)
    print(f"  Dice Coefficient : {dice:.4f}")
    print(f"  IoU (Jaccard)    : {iou:.4f}")
    print(f"  F1 Score         : {f1:.4f}")
    print(f"  ROC-AUC          : {auc:.4f}")
    print(f"  Best threshold   : {threshold:.2f}")
    print(f"  TTA enabled      : {CFG['use_tta']}")
    print(f"  Parameters       : {model.count_params():,}")
    print(f"  Input channels   : {SPECTRAL_CHANNELS} (6 spectral + NDVI + EVI + SAVI)")
    print(f"  Split strategy   : Stratified random 80/20")
    print(f"  Outputs saved to : {CFG['output_dir']}")
    print("=" * 57)

print_summary(y_true_val, y_soft_val, best_threshold)

## 20. Doctor Testing UI

In [ ]:
def run_doctor_ui(model):

    title_html = widgets.HTML("""
    <div style="
        background:linear-gradient(135deg,#0d3b1e 0%,#145a32 60%,#1e8449 100%);
        border-radius:12px;padding:24px 32px;margin-bottom:16px;
        font-family:Georgia,serif;color:white;
        box-shadow:0 4px 20px rgba(0,0,0,0.4);">
        <div style="font-size:22px;font-weight:bold;letter-spacing:1px;">
            🌲 Forest Loss Detection v4 — Researcher Review Tool
        </div>
        <div style="font-size:13px;opacity:0.85;margin-top:6px;">
            Upload Sentinel-2 patch files (.npy) for T1 (2021) and T2 (2023).
            Model detects deforested pixels and estimates area lost in hectares.
        </div>
    </div>
    """)

    t1_upload = widgets.FileUpload(
        accept=".npy", multiple=False,
        description="📂 T1 patch (2021)",
        layout=widgets.Layout(width="340px"),
        style={"description_width": "140px"}
    )
    t2_upload = widgets.FileUpload(
        accept=".npy", multiple=False,
        description="📂 T2 patch (2023)",
        layout=widgets.Layout(width="340px"),
        style={"description_width": "140px"}
    )
    threshold_slider = widgets.FloatSlider(
        value=best_threshold, min=0.1, max=0.9, step=0.01,
        description="Threshold:",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="380px"),
        readout_format=".2f",
    )
    tta_toggle = widgets.ToggleButton(
        value=True, description="TTA  ON", button_style="success",
        layout=widgets.Layout(width="130px"),
    )
    def on_tta_change(change):
        tta_toggle.description  = "TTA  ON"  if change["new"] else "TTA  OFF"
        tta_toggle.button_style = "success"  if change["new"] else "warning"
    tta_toggle.observe(on_tta_change, names="value")

    analyse_btn = widgets.Button(
        description="  🔍  ANALYSE PATCH",
        layout=widgets.Layout(width="220px", height="44px"),
    )
    analyse_btn.style.button_color = "#1e8449"
    status_label = widgets.HTML("")
    output_area  = widgets.Output()

    def on_analyse(_):
        with output_area:
            clear_output(wait=True)
            if not t1_upload.value or not t2_upload.value:
                status_label.value = (
                    "<span style='color:#e74c3c;font-weight:bold;'>"
                    "⚠ Please upload both T1 and T2 .npy files first.</span>")
                return

            status_label.value = "<span style='color:#2980b9;'>⏳ Loading patches...</span>"
            try:
                t1_bytes = list(t1_upload.value.values())[0]["content"]
                t2_bytes = list(t2_upload.value.values())[0]["content"]
                img1 = np.load(io.BytesIO(bytes(t1_bytes))).astype(np.float32)
                img2 = np.load(io.BytesIO(bytes(t2_bytes))).astype(np.float32)
            except Exception as e:
                status_label.value = (
                    f"<span style='color:#e74c3c;'>❌ File error: {e}</span>")
                return

            if img1.ndim == 3 and img1.shape[0] == CFG["in_channels"]:
                img1 = np.transpose(img1, (1, 2, 0))
                img2 = np.transpose(img2, (1, 2, 0))

            for c in range(img1.shape[-1]):
                mn, mx = np.percentile(img1[...,c], (2, 98))
                img1[...,c] = (img1[...,c] - mn) / (mx - mn + 1e-6)
            for c in range(img2.shape[-1]):
                mn, mx = np.percentile(img2[...,c], (2, 98))
                img2[...,c] = (img2[...,c] - mn) / (mx - mn + 1e-6)

            img1 = add_vegetation_indices(img1)
            img2 = add_vegetation_indices(img2)
            diff = np.abs(img2 - img1)

            status_label.value = (
                f"<span style='color:#2980b9;'>⏳ Running inference"
                f"{'  +TTA (8 passes)' if tta_toggle.value else ''}...</span>")

            prob_map  = tta_predict(
                model,
                img1[np.newaxis], img2[np.newaxis], diff[np.newaxis],
                use_tta=tta_toggle.value
            )[0,:,:,0]
            thr       = threshold_slider.value
            pred_mask = (prob_map > thr).astype(np.uint8)
            area_ha   = pred_mask.sum() * (CFG["pixel_res_m"] ** 2) / 10_000
            loss_pct  = 100.0 * pred_mask.mean()
            conf_mean = (float(prob_map[pred_mask==1].mean())
                         if pred_mask.sum() > 0 else 0.0)

            fig = plt.figure(figsize=(24, 7), facecolor="#f4f6f7")
            gs  = gridspec.GridSpec(1, 6, figure=fig,
                                    wspace=0.04, left=0.01, right=0.99,
                                    top=0.88, bottom=0.05)
            panel_data = [
                (make_rgb_composite(img1,(2,1,0)), None,       "T1 2021 — True Colour RGB"),
                (make_rgb_composite(img2,(2,1,0)), None,       "T2 2023 — True Colour RGB"),
                (make_rgb_composite(img2,(3,2,1)), None,       "T2 2023 — False-colour IR"),
                (prob_map,                         "RdYlGn_r", "Probability Map (soft)"),
                (pred_mask,                        "Reds",     f"Binary Mask (thr={thr:.2f})"),
            ]
            for idx_, (data, cmap, title) in enumerate(panel_data):
                ax = fig.add_subplot(gs[0, idx_])
                if cmap:
                    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
                    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                else:
                    ax.imshow(data)
                ax.set_title(title, fontsize=10, fontweight="bold", pad=5)
                ax.axis("off")

            ax_ov = fig.add_subplot(gs[0, 5])
            ax_ov.imshow(make_rgb_composite(img2,(2,1,0)))
            overlay = np.zeros((*pred_mask.shape, 4), dtype=np.float32)
            overlay[pred_mask==1] = [1.0, 0.1, 0.1, 0.55]
            ax_ov.imshow(overlay)
            ax_ov.set_title("T2 + Prediction Overlay",
                            fontsize=10, fontweight="bold", pad=5)
            ax_ov.axis("off")

            sev_color = ("#c0392b" if loss_pct > 5 else
                         "#e67e22" if loss_pct > 1 else "#27ae60")
            fig.suptitle(
                f"Forest Loss Analysis  |  Area lost: {area_ha:.2f} ha  |  "
                f"{loss_pct:.2f}% of patch  |  Mean confidence: {conf_mean:.3f}",
                fontsize=13, fontweight="bold", color=sev_color, y=0.97
            )
            save_path = os.path.join(CFG["output_dir"], "doctor_result_v4.png")
            plt.savefig(save_path, dpi=160, bbox_inches="tight", facecolor="#f4f6f7")
            plt.show()

            sev_label = (
                "🔴 HIGH — significant deforestation detected"  if loss_pct > 5 else
                "🟠 MODERATE — localised forest loss detected"  if loss_pct > 1 else
                "🟢 LOW — minimal or no deforestation detected"
            )
            status_label.value = f"""
            <div style="background:#f0f9f4;border-left:4px solid #1e8449;
                        padding:14px 18px;border-radius:6px;
                        font-family:monospace;font-size:13px;margin-top:10px;">
                <b>RESULT SUMMARY</b><br>
                ▸ Severity    : {sev_label}<br>
                ▸ Area lost   : <b>{area_ha:.2f} ha</b><br>
                ▸ Patch cover : {loss_pct:.2f}% of 256×256 patch<br>
                ▸ Mean conf.  : {conf_mean:.3f}<br>
                ▸ TTA used    : {'Yes (8 passes)' if tta_toggle.value else 'No'}<br>
                ▸ Threshold   : {thr:.2f}<br>
                ▸ Saved to    : {save_path}
            </div>"""

    analyse_btn.on_click(on_analyse)
    display(widgets.VBox([
        title_html,
        widgets.HBox([t1_upload, t2_upload],
                     layout=widgets.Layout(gap="20px")),
        widgets.HBox([threshold_slider, tta_toggle, analyse_btn],
                     layout=widgets.Layout(gap="20px", align_items="center")),
        status_label,
        output_area,
    ], layout=widgets.Layout(padding="10px")))


run_doctor_ui(model)